# Practical 8: Family Relations (Prolog)

Transcribed from `Notes_260912_182345.PDF`. The facts and rules remain in Prolog. This notebook uses the Python 3 kernel to run SWI-Prolog; no special Jupyter kernel is needed.

Install SWI-Prolog on the machine running JupyterLab. On Ubuntu/Debian, run `sudo apt install swi-prolog` in a terminal. Ensure `swipl` is on PATH, then run the cells below.

The PDF's final three example answers contradict its facts: `uncle(vikram,varun)`, `aunt(meera,varun)`, and `cousin(karan,diya)` are false. The supplied facts/rules are preserved. Definitions and a family-tree drawing were not supplied in this document.


Names have been changed consistently; all genders and parent-child relationships match the supplied document. The displayed query results are computed from these facts, not copied from the PDF.


In [ ]:
prolog_source = r'''
% ---------- Facts ----------

male(arjun).
male(rohan).
male(vikram).
male(karan).
male(aman).
male(raj).
male(varun).

female(priya).
female(neha).
female(riya).
female(ananya).
female(kavya).
female(meera).
female(diya).

% parent(Parent, Child)
parent(arjun, rohan).
parent(priya, rohan).
parent(arjun, riya).
parent(priya, riya).
parent(rohan, karan).
parent(neha, karan).
parent(rohan, ananya).
parent(neha, ananya).
parent(vikram, kavya).
parent(meera, kavya).
parent(vikram, aman).
parent(meera, aman).
parent(karan, varun).
parent(ananya, diya).

% ---------- Rules ----------

% Father
father(X, Y) :-
    male(X),
    parent(X, Y).

% Mother
mother(X, Y) :-
    female(X),
    parent(X, Y).

% Grandfather
grandfather(X, Y) :-
    male(X),
    parent(X, Z),
    parent(Z, Y).

% Grandmother
grandmother(X, Y) :-
    female(X),
    parent(X, Z),
    parent(Z, Y).

% Brother
brother(X, Y) :-
    male(X),
    parent(P, X),
    parent(P, Y),
    X \= Y.

% Sister
sister(X, Y) :-
    female(X),
    parent(P, X),
    parent(P, Y),
    X \= Y.

% Uncle
uncle(X, Y) :-
    male(X),
    parent(P, Y),
    brother(X, P).

% Aunt
aunt(X, Y) :-
    female(X),
    parent(P, Y),
    sister(X, P).

% Nephew
nephew(X, Y) :-
    male(X),
    (uncle(Y, X) ; aunt(Y, X)).

% Niece
niece(X, Y) :-
    female(X),
    (uncle(Y, X) ; aunt(Y, X)).

% Cousin
cousin(X, Y) :-
    parent(P1, X),
    parent(P2, Y),
    (brother(P1, P2) ; sister(P1, P2)),
    X \= Y.
'''

In [ ]:
import shutil
import subprocess
import tempfile
from pathlib import Path

swipl = shutil.which("swipl")
if swipl is None:
    raise RuntimeError("Install SWI-Prolog and ensure swipl is on PATH, then rerun this cell.")

queries = ['father(arjun, rohan)', 'mother(priya, riya)', 'grandfather(arjun, karan)', 'grandmother(priya, ananya)', 'brother(riya, rohan)', 'sister(riya, rohan)', 'uncle(vikram, varun)', 'aunt(meera, varun)', 'cousin(karan, diya)']

with tempfile.TemporaryDirectory() as folder:
    source = Path(folder) / "family.pl"
    source.write_text(prolog_source, encoding="utf-8")
    for query in queries:
        goal = f"(({query}) -> writeln(true) ; writeln(false))"
        result = subprocess.run(
            [swipl, "-q", "-s", str(source), "-g", goal, "-t", "halt"],
            capture_output=True, text=True, check=True, timeout=10
        )
        print(f"?- {query}.\n{result.stdout.strip()}.\n")


?- father(arjun, rohan).
true.

?- mother(priya, riya).
true.

?- grandfather(arjun, karan).
true.

?- grandmother(priya, ananya).
true.

?- brother(riya, rohan).
false.

?- sister(riya, rohan).
true.

?- uncle(vikram, varun).
false.

?- aunt(meera, varun).
false.

?- cousin(karan, diya).
false.

